In [1]:
""" Create the observational data set after controlling for pretreatment effects
    "extract_obs_productivity.ipynb" only applies it on biomass

    Because of the considerable pre-treatment effect reported in Hanson et al. 2025,
    we also remove the pre-treatment effect from AGNPP of the trees

    Include Year in the linear regression model and keep only one pretreatment term
        to avoid over-subtraction of pretreatment effect

    Hanson, P. J., Griffiths, N. A., Salmon, V. G., Birkebak, J. M., Warren, J. M., Phillips, J. R., et al. (2025). Peatland Plant Community Changes in Annual Production and Composition Through 8 Years of Warming Manipulations Under Ambient and Elevated CO2 Atmospheres. Journal of Geophysical Research: Biogeosciences, 130(2), e2024JG008511. https://doi.org/10.1029/2024JG008511
"""
import pandas as pd
import os
import numpy as np
import xarray as xr
import statsmodels.formula.api as smf
from utils.constants import *

In [2]:
#############################################################
# Collect all the observations into a single pandas dataframe
#############################################################

plot_list = [f'P{p:02g}' for p in chamber_list_complete]
obs_data = pd.DataFrame(np.nan, 
                        index = pd.MultiIndex.from_product([plot_list, range(2015, 2024)], names=['Plot','Year']), 
                        columns = ['eCO2', 'Tair', 
                                   'AGNPP_Spruce', 'AGNPP_Tamarack', 'AGNPP_Shrub', 'NPP_moss', # gC m-2 yr-1
                                   'BGNPP_TreeShrub', # fine root NPP, gC m-2 yr-1
                                   'NPP', # AGNPP tree + shrub + moss + (fine root) BGNPP tree shrub
                                   'HR', 'NEE']) # 'BGtoAG_TreeShrub', # fine root NPP to AGNPP ratio
                                   ##'AGBiomass_Spruce', 'AGBiomass_Tamarack', 'AGBiomass_Shrub', # gC m-2
                                   ##'AGNPPtoBiomass_Spruce', 'AGNPPtoBiomass_Tamarack', 'AGNPPtoBiomass_Shrub', # NPP to Biomass ratio, yr-1
obs_data['eCO2'] = [chamber_levels_complete[pp[1:]][1] for pp in obs_data.index.get_level_values(0)]

# -------------------------
# (1) Paul's data 2016-2021
# -------------------------
obs_paul = pd.read_excel(os.path.join(os.environ['PROJDIR'], 'ELM_Nutrients', 'input', 'SPRUCE C Budget Summary 28Apr2022EXP.xlsx'),
                        sheet_name="DataForPythonRead", skiprows=1,  engine="openpyxl")
obs_paul = obs_paul.loc[obs_paul["Year"] != 2020, :]
obs_paul = obs_paul.set_index(['Plot', 'Year']).sort_index()

intersect_paul = obs_data.index.intersection(obs_paul.index)
rename_dict = {'Mean Annual Temp. at 2 m': 'Tair', 'ANPP Shrub (~50%C)': 'AGNPP_Shrub',
               'NPP Sphag.': 'NPP_moss', 'BNPP Tree & Shrub': 'BGNPP_TreeShrub', 
               'RHCO2': 'HR', 'NCE': 'NEE'}
for col in rename_dict.keys():
    obs_data.loc[intersect_paul, rename_dict[col]] = obs_paul[col]
    if col == 'RHCO2' or col == 'NCE':
        obs_data[rename_dict[col]] *= -1

# -------------------------------------------------
# (2) Pretreatment data from Verity 2016-2021 table
# -------------------------------------------------
obs_verity = pd.read_csv(os.path.join(os.environ['PROJDIR'], 'ELM_Nutrients', 'input', 'SalmonSPRUCE_2016to2021_AbovegroundPFT_CNPbudget_20240208.csv'))
# match by plot and year to temperature
obs_verity['Plot'] = [f'P{p:02d}' for p in obs_verity['Plot']]
obs_verity = obs_verity.set_index(['Plot', 'Year', 'PFT']).unstack()
obs_verity = obs_verity.loc[:, (slice(None), ['Sphagnum', 'evergreen conifer', 'deciduous conifer', 'shrub'])]
# 'ABGbiomass_gCperm2', 'ABGnpp_gCperm2peryear', 'Pretrt_ABGbiomass_gCperm2', 
obs_verity = obs_verity.loc[(slice(None), 2016), 'Pretrt_ABGnpp_gCperm2peryear']
obs_verity.index = obs_verity.index.droplevel(1)

for plot in obs_verity.index:
    obs_data.loc[(plot,2015), 'AGNPP_Spruce'] = obs_verity.loc[plot, 'evergreen conifer']
    obs_data.loc[(plot,2015), 'AGNPP_Tamarack'] = obs_verity.loc[plot, 'deciduous conifer']
    obs_data.loc[(plot,2015), 'AGNPP_Shrub'] = obs_verity.loc[plot, 'shrub']
    obs_data.loc[(plot,2015), 'NPP_moss'] = obs_verity.loc[plot, 'Sphagnum']

# --------------------------------------------------------------------
# (3) Posttreatment data from Verity 2016-2023 table
#     (not mixing in the previous table for consistency)
# --------------------------------------------------------------------
obs_verity2 = pd.read_csv(os.path.join(os.environ['PROJDIR'], 'ELM_Nutrients', 'input', 'SPRUCE_AGB&ANPP_CNP_PerTissue_20260123.csv'))
obs_verity2['Plot'] = [f'P{p:02g}' for p in obs_verity2['Plot']]
# sum to AGNPP for vascular plants and NPP for Sphagnum
obs_verity2 = obs_verity2.loc[obs_verity2['PFT'].isin(['Picea','Larix','Shrub','Sphagnum'])
                              ].set_index(['Plot','Year','PFT','Tissue']
                              )['SumANPP_gCperm2peryear'].unstack().sum(axis=1, min_count=1).unstack()

intersect_verity2 = obs_data.index.intersection(obs_verity2.index)
obs_data.loc[intersect_verity2, 'AGNPP_Spruce'] = obs_verity2.loc[intersect_verity2, 'Picea']
obs_data.loc[intersect_verity2, 'AGNPP_Tamarack'] = obs_verity2.loc[intersect_verity2, 'Larix']
obs_data.loc[intersect_verity2, 'AGNPP_Shrub'] = obs_verity2.loc[intersect_verity2, 'Shrub']
obs_data.loc[intersect_verity2, 'NPP_moss'] = obs_verity2.loc[intersect_verity2, 'Sphagnum']

# --------------------------------------------------------------------
# (4) Add observed annual mean temperature from SPRUCE forcing data
# --------------------------------------------------------------------
tair = {}
for plot in plot_list:
    hr = xr.open_dataset(os.path.join(os.environ['E3SM_ROOT'], 'inputdata', 'atm', 'datm7', 'CLM1PT_data',
                                        'SPRUCE_data', f'plot{plot[1:]}', 'all_hourly.nc'))
    tt = hr['TBOT'][0, :].resample(DTIME='1YE').mean()
    tair[plot] = pd.Series(tt.values, index = tt['DTIME'].to_index().year) - 273.15
    hr.close()
tair = pd.DataFrame(tair).unstack()
tair.index.names = ['Plot','Year']

# There is minimal difference between Paul's tair and
# ELM forcing tair. Use ELM forcing for consistency
### pd.concat([obs_data['Tair], tair], axis = 1).plot()
intersect = obs_data.index.intersection(tair.index)
obs_data.loc[intersect, 'Tair'] = tair.loc[intersect]

In [3]:
# Look at the outcome
obs_data

eCO2       Tair  AGNPP_Spruce  AGNPP_Tamarack  AGNPP_Shrub  \
Plot Year                                                               
P07  2015     0   5.589600           NaN             NaN          NaN   
     2016     0   6.247101     84.595841       13.980000    45.660269   
     2017     0   5.114349     69.600798       12.720000    57.932788   
     2018     0   4.112640     82.770000       19.685464    91.231935   
     2019     0   3.945648     82.930000       18.380000    83.343799   
...         ...        ...           ...             ...          ...   
P10  2019   500  14.116791    123.600000       45.870000   167.289313   
     2020   500  15.851593    144.430000       47.850000          NaN   
     2021   500  16.390411    108.930000       45.260000   204.348328   
     2022   500  14.109833    126.190000       36.490000          NaN   
     2023   500  16.905151    161.228939       65.040000   103.571692   

             NPP_moss  BGNPP_TreeShrub  NPP          HR         NEE  
Plot Year                                                            
P07  2015         NaN              NaN  NaN         NaN         NaN  
     2016   95.836556          9.40000  NaN  369.900000  251.706491  
     2017  126.370266         10.70000  NaN  352.000000  140.450021  
     2018         NaN         10.05000  NaN  356.900000  132.419029  
     2019         NaN         10.37500  NaN  305.881296   57.562158  
...               ...              ...  ...         ...         ...  
P10  2019    7.646592        125.27500  NaN  608.049941  556.482299  
     2020    5.111946              NaN  NaN         NaN         NaN  
     2021    8.128313        123.01875  NaN  685.181196  296.032065  
     2022         NaN              NaN  NaN         NaN         NaN  
     2023         NaN              NaN  NaN         NaN         NaN  

[99 rows x 10 columns]

In [4]:
# Use linear regression to remove pre-treatment effect on the
# aboveground biomass & aboveground NPP of trees
# 
# Subtract out all three terms that include pretreatment effect if significant

def create_Xy(df, varname):
    """ Covariates: eCO2, Tair, Year, Pre-treatment varname
        Target: Post treatment varname
    """
    df = df.sort_index()
    years = df.index.get_level_values('Year').unique().sort_values()
    first_year = years[0]
    later_years = years[1:]

    # Target: colname for years 2 to last
    y_df = df.loc[df.index.get_level_values('Year').isin(later_years), [varname]].rename(columns={varname: 'Postrt'})

    # Covariate 1: colname value in first year (per Plot)
    first_year_vals = df.xs(first_year, level='Year')[[varname]].rename(columns={varname: 'Pretrt'})

    # Covariates 2 & 3: col1, col2 for years 2 to last
    cov_df = df.loc[df.index.get_level_values('Year').isin(later_years), ['eCO2','Tair']]

    # Combine
    X_df = cov_df.copy()
    X_df = X_df.join(first_year_vals, on='Plot')
    X_df['Year'] = X_df.index.get_level_values('Year')

    # Align
    X = pd.concat([X_df, y_df], axis = 1, join = 'inner')

    return X


for prefix in ['AGNPP']: # 'AGBiomass', 
    for pft in ['Spruce', 'Tamarack', 'Shrub']:
        varname = f'{prefix}_{pft}'

        # Form the dataset
        df = create_Xy(obs_data, varname)

        # Full model
        full_formula = ("Postrt ~ eCO2 + Tair + Year + Pretrt + eCO2:Tair + eCO2:Year")
        full_mod = smf.ols(full_formula, data=df).fit()

        # Drop insignificant terms and refit
        keepers = [
            term for term, p in full_mod.pvalues.items()
            if (term == "Intercept") or (p <= 0.05)
        ]

        print(varname, keepers)

        # No pretreatment effect, then subtract nothing
        if not ('Pretrt' in keepers):
            obs_data.loc[:, varname] = obs_data[varname].values

            # It is not possible to perform a sensible estimation of this uncertainty!!!!!!!
            # Just go with proportional to Paul's numbers

            ### Calculate uncertainty in data as simple standard deviation in the
            ### pretreatment values
            ###std0 = obs_data.loc[obs_data['Year'] == 2016, :][(varpre, pft2)].std()
            ###print(varname, pft, 'aCO2 std=', std0, 'eCO2 std=', std0)

            continue

        # Has pretreatment effect, fit reduced model
        reduced_formula = "Postrt ~ " + " + ".join(k for k in keepers if k != "Intercept")
        results = smf.ols(reduced_formula, data=df).fit()

        print(results.summary())

        # Subtract the pretreatment effect if significant
        # (add back the mean of the pretreatment level)
        if 'Pretrt' in results.pvalues.index and results.pvalues['Pretrt'] <= 0.05:
            baseline = results.params['Pretrt'] * (df['Pretrt'] - df['Pretrt'].mean())
            obs_data.loc[df.index, varname] = df['Postrt'] - baseline

        # It is not possible to perform a sensible estimation of this uncertainty!!!!!!!
        # Just go with proportional to Paul's numbers

        ## Print the uncertainty in the subtracted value
        ## - ambient CO2
        #var0 = np.sqrt(results.bse['Intercept']**2 + \
        #       results.bse['Pretrt']**2 * obs_data[(varpre, pft2)].mean() + \
        #       results.params['Pretrt']**2 * obs_data[(varpre, pft2)].var())
        #print(varname, pft, gamma_list, var0)

# Remove pretreatment
obs_data = obs_data.loc[obs_data.index.get_level_values('Year') != 2015, :]

AGNPP_Spruce ['Intercept', 'Pretrt', 'eCO2:Tair']
                            OLS Regression Results                            
Dep. Variable:                 Postrt   R-squared:                       0.254
Model:                            OLS   Adj. R-squared:                  0.235
Method:                 Least Squares   F-statistic:                     13.13
Date:                Thu, 26 Feb 2026   Prob (F-statistic):           1.24e-05
Time:                        21:16:12   Log-Likelihood:                -423.50
No. Observations:                  80   AIC:                             853.0
Df Residuals:                      77   BIC:                             860.1
Df Model:                           2                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
In

In [5]:
#for pft,pft2 in zip(['Spruce', 'Tamarack', 'Shrub'], 
#                    ['evergreen conifer','deciduous conifer','shrub']):
#    obs_data2.loc[:, f'AGNPPtoBiomass_{pft}'] = \
#        obs_data[('ABGnpp_gCperm2peryear', pft2)].values / \
#        obs_data[('ABGbiomass_gCperm2', pft2)].values

#obs_data2.loc[:, 'BGtoAG_TreeShrub'] = \
#    obs_data2.loc[:, 'BGNPP_TreeShrub'].values / \
#    (obs_data2.loc[:, 'AGNPP_Spruce'] + obs_data2.loc[:, 'AGNPP_Tamarack'] + \
#     obs_data2.loc[:, 'AGNPP_Shrub']).values


############################
# Final processing and save
############################
# Exclude tree AGNPP in 2023 because of clipping
obs_data.loc[obs_data.index.get_level_values('Year') == 2023, 'AGNPP_Spruce'] = np.nan
obs_data.loc[obs_data.index.get_level_values('Year') == 2023, 'AGNPP_Tamarack'] = np.nan

#
obs_data.loc[:, 'NPP'] = (obs_data.loc[:, 'AGNPP_Spruce'] + obs_data.loc[:, 'AGNPP_Tamarack'] + \
     obs_data.loc[:, 'AGNPP_Shrub'] + obs_data.loc[:, 'BGNPP_TreeShrub'] + \
     obs_data.loc[:, 'NPP_moss']).values

path_out = os.path.join(os.environ['PROJDIR'], 'ELM_Nutrients', 'output', 'extract')
obs_data.to_csv(os.path.join(path_out, 'extract_obs_productivity.csv'))

obs_data

eCO2       Tair  AGNPP_Spruce  AGNPP_Tamarack  AGNPP_Shrub  \
Plot Year                                                               
P07  2016     0   6.247101           NaN             NaN    45.660269   
     2017     0   5.114349           NaN             NaN    57.932788   
     2018     0   4.112640           NaN             NaN    91.231935   
     2019     0   3.945648           NaN             NaN    83.343799   
     2020     0   5.592621           NaN             NaN          NaN   
...         ...        ...           ...             ...          ...   
P10  2019   500  14.116791    140.813337       59.087195   167.289313   
     2020   500  15.851593    161.643337       61.067195          NaN   
     2021   500  16.390411    126.143337       58.477195   204.348328   
     2022   500  14.109833    143.403337       49.707195          NaN   
     2023   500  16.905151           NaN             NaN   103.571692   

             NPP_moss  BGNPP_TreeShrub         NPP          HR         NEE  
Plot Year                                                                   
P07  2016   95.836556          9.40000         NaN  369.900000  251.706491  
     2017  126.370266         10.70000         NaN  352.000000  140.450021  
     2018         NaN         10.05000         NaN  356.900000  132.419029  
     2019         NaN         10.37500         NaN  305.881296   57.562158  
     2020         NaN              NaN         NaN         NaN         NaN  
...               ...              ...         ...         ...         ...  
P10  2019    7.646592        125.27500  500.111436  608.049941  556.482299  
     2020    5.111946              NaN         NaN         NaN         NaN  
     2021    8.128313        123.01875  520.115922  685.181196  296.032065  
     2022         NaN              NaN         NaN         NaN         NaN  
     2023         NaN              NaN         NaN         NaN         NaN  

[88 rows x 10 columns]

In [6]:
obs_data.groupby('eCO2').mean()

,Tair,AGNPP_Spruce,AGNPP_Tamarack,AGNPP_Shrub,NPP_moss,BGNPP_TreeShrub,NPP,HR,NEE
eCO2,,,,,,,,,
0,10.286420,94.903008,41.423639,79.463752,87.24286,75.16375,391.261515,429.644529,174.324416
500,11.138966,94.135726,32.284974,108.671270,63.87815,79.21125,384.876521,460.730872,210.644919
